# DBRepo Ontological Unit/Concept Mapping

This notebook maps every numeric column in the schema to an ontological concept and writes the mapping to DBRepo column metadata via `client.update_table_column()`.

## Numeric columns in scope

| Table | Column | Meaning |
|---|---|---|
| `unemployment` | `value` | Number of unemployed persons (count) |
| `unemployment` | `density` | Unemployed persons per 1,000 inhabitants (ratio) |
| `tourism` | `value` | Number of overnight stays (count) |
| `tourism` | `density` | Overnight stays per 1,000 inhabitants (ratio) |
| `measurement_info` | `population_avg` | Average population in a specific district (count) |

Non-numeric columns (`district_id`, `measurement_id`, `nuts_code`, `district_code`, `reference_date`, `gender`) are surrogate keys, codes, dates, or categoricals that carry no physical unit and are excluded.

---

## Ontology selection and justification

### Selected ontology: OMG Commons Quantities and Units (cmns-qtu)
**Namespace:** `https://www.omg.org/spec/Commons/QuantitiesAndUnits/`  
**Serialisation:** `https://www.omg.org/spec/Commons/QuantitiesAndUnits.ttl`

The OMG Commons Quantities and Units ontology is chosen because it explicitly defines two concepts that are precise matches for our columns:
1. **`cmns-qtu:Total`**: defined as "sum of the values for some characteristic of all units", matching the `value` and `population_avg` count columns.
2. **`cmns-qtu:Ratio`**: defined as "proportional relationship between two different quantity values", synonymous with "rate", matching the `density` columns (overnight stays or unemployed persons per 1,000 inhabitants).

### Unit URIs used

| Column(s) | Unit | Unit URI |
|---|---|---|
| `unemployment.value`, `tourism.value`, `measurement_info.population_avg` | Total (sum of values for a characteristic) | `https://www.omg.org/spec/Commons/QuantitiesAndUnits/Total` |
| `unemployment.density`, `tourism.density` | Ratio (proportional relationship, rate) | `https://www.omg.org/spec/Commons/QuantitiesAndUnits/Ratio` |

## 1 · Imports & connection

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient

load_dotenv()
load_dotenv(os.path.join(os.path.dirname(__file__), '..', 'config', '.env'))

ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
DATABASE_ID = "412fb0ce-5299-4d0e-a271-4641b1365b8a"
USERNAME = os.getenv("DBREPO_USERNAME")
PASSWORD = os.getenv("DBREPO_PASSWORD")

client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)
print("Connected as:", client.whoami())

Binw3g
Connected as: Binw3g


## 2 · Build column -> unit URI mapping

We fetch all tables and build a lookup of `(table_name, column_name) -> column_id` to use with `update_table_column()`.

In [38]:
# OMG Commons Quantities and Units concept URIs
# Namespace: https://www.omg.org/spec/Commons/QuantitiesAndUnits/
OMG_TOTAL = "https://www.omg.org/spec/Commons/QuantitiesAndUnits/Total"
OMG_RATIO = "https://www.omg.org/spec/Commons/QuantitiesAndUnits/Ratio"

# Mapping: (table_name, column_name) -> unit URI
CONCEPT_MAP: dict[tuple[str, str], str] = {
    ("unemployment", "value"): OMG_TOTAL,  # count of unemployed persons
    ("unemployment", "density"): OMG_RATIO,  # unemployed per 1,000 inhabitants
    ("tourism", "value"): OMG_TOTAL,  # count of overnight stays
    ("tourism", "density"): OMG_RATIO,  # overnight stays per 1,000 inhabitants
    ("measurement_info", "population_avg"): OMG_TOTAL,  # average population count in a specific district
}

print("Concept map defined:")
for (table, col), uri in CONCEPT_MAP.items():
    short = uri.split("/")[-1]
    print(f"  {table}.{col} -> {short}")

Concept map defined:
  unemployment.value -> Total
  unemployment.density -> Ratio
  tourism.value -> Total
  tourism.density -> Ratio
  measurement_info.population_avg -> Total


In [39]:
# Build (table_name, col_name) -> (table_id, column_id) lookup
tables = client.get_tables(DATABASE_ID)
col_id_map: dict[tuple[str, str], tuple[str, str]] = {}
col_concept_map: dict[tuple[str, str], str] = {}

for table in tables:
    full_table = client.get_table(DATABASE_ID, table.id)
    for col in full_table.columns:
        col_id_map[(table.name, col.name)] = (table.id, col.id)
        col_concept_map[(table.name, col.name)] = col.concept_uri

print(f"Resolved {len(col_id_map)} columns across {len(tables)} tables.")

# Verify all mapped columns are present
missing = [k for k in CONCEPT_MAP if k not in col_id_map]
if missing:
    raise RuntimeError(f"Columns not found in DBRepo: {missing}")
print("✓ All target columns found.")

Resolved 14 columns across 4 tables.
✓ All target columns found.


## 3 · Apply concept mappings

For each column in `CONCEPT_MAP` we call `client.update_table_column()` to register the unit URI.

In [40]:
AUTH = (USERNAME, PASSWORD)
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
BASE = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}"

def apply_concept(table_name: str, column_name: str, concept_uri: str, unit_uri: str) -> None:
    (table_id, column_id) = col_id_map[(table_name, column_name)]
    short_uri = unit_uri.split("/")[-1]

    client.update_table_column(
        database_id=DATABASE_ID,
        table_id=table_id,
        column_id=column_id,
        concept_uri=concept_uri,
        unit_uri=unit_uri
    )
    print(f"  ✓ {table_id}.{column_id:<20} -> {short_uri}")

print("=== Applying concept mappings ===")
for (table_name, col_name), unit_uri in CONCEPT_MAP.items():
    concept_uri = col_concept_map.get((table_name, col_name))
    apply_concept(table_name, col_name, concept_uri, unit_uri)

print("\n✓ All mappings applied.")

=== Applying concept mappings ===
  ✓ 5a3f74b9-8a62-4f33-9486-6c6cd3d34cd2.e198310d-fed8-4550-96aa-0337509fcb3f -> Total
  ✓ 5a3f74b9-8a62-4f33-9486-6c6cd3d34cd2.f8c5c11a-3bdf-411b-9e9d-2086ba02ffc0 -> Ratio
  ✓ fa5fe819-9b20-422a-a6fe-299ba0043d87.895a12d5-1058-4f41-b4f5-3af1cdda3422 -> Total
  ✓ fa5fe819-9b20-422a-a6fe-299ba0043d87.dd855875-fc15-42ea-9536-f88368054759 -> Ratio
  ✓ 5d6d28a7-f211-4bc4-9ada-94450cd0fe6f.e78e310d-1409-4b62-b078-e677e2adac1b -> Total

✓ All mappings applied.


## 4 · Verify mappings were persisted

Re-fetch each table and confirm the `unit_uri` is present on the expected columns.

In [43]:
rows = []
for table in client.get_tables(DATABASE_ID):
    full = client.get_table(DATABASE_ID, table.id)
    for col in full.columns:
        key = (table.name, col.name)
        
        if key in CONCEPT_MAP:
            unit_val = getattr(col, 'unit_uri', None)
            expected = CONCEPT_MAP[key]
            status = "✓" if (unit_val and expected.split("/")[-1] in str(unit_val)) else "✗"
            
            rows.append({
                "table":    table.name,
                "column":   col.name,
                "expected": expected.split("/")[-1],
                "stored":   unit_val,
                "ok":       status,
            })

df_check = pd.DataFrame(rows)
print(df_check.to_string(index=False))

failed = df_check[df_check["ok"] == "✗"]
if not failed.empty:
    print(f"\n⚠ {len(failed)} column(s) did not verify, check the stored values above.")
else:
    print("\n✓ All unit mappings verified successfully.")

           table         column expected                                                    stored ok
    unemployment          value    Total https://www.omg.org/spec/Commons/QuantitiesAndUnits/Total  ✓
    unemployment        density    Ratio https://www.omg.org/spec/Commons/QuantitiesAndUnits/Ratio  ✓
         tourism          value    Total https://www.omg.org/spec/Commons/QuantitiesAndUnits/Total  ✓
         tourism        density    Ratio https://www.omg.org/spec/Commons/QuantitiesAndUnits/Ratio  ✓
measurement_info population_avg    Total https://www.omg.org/spec/Commons/QuantitiesAndUnits/Total  ✓

✓ All unit mappings verified successfully.
